In [2]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="2"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict
from peft import LoraConfig, get_peft_model
import numpy as np


import prompts

In [3]:
## prepare data

# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

#read qq data
qq_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv')
qq_ds = Dataset.from_pandas(qq_df.loc[:,['question','follow_up_questions']])

In [40]:
def embedd_and_save_evidence(evidence_ls):
    # calculate embeddings for all data
    evidence_embeddings = model.encode(evidence_ls)
    evidence_embeddings = np.array(evidence_embeddings)
    print(evidence_embeddings.shape)
    np.save(embedding_file, evidence_embeddings)
    
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

embedding_file = '/raid/deallab/SF_RAG_Data/ASQA/train_embeddings.npy'
re_embedd = True

if not os.path.exists(embedding_file) or re_embedd:    
    embedd_and_save_evidence(evidence_ls)

evidence_embeddings = np.load(embedding_file)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device)

(178876, 384)
(178876, 384)


In [56]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:20]
    # print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [43]:
# load generative model
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
model_gen.resize_token_embeddings(len(tokenizer_gen))

# peft_config = LoraConfig(
#         target_modules=[ "v_proj", "q_proj", "up_proj", "o_proj", "k_proj", "down_proj", "gate_proj" ], 
#         inference_mode=False, 
#         r=4, 
#         lora_alpha=32, 
#         lora_dropout=0.1,    
#         task_type="CAUSAL_LM",        
#     )

# LMmodel = get_peft_model(model_gen, peft_config)

# LMmodel.print_trainable_parameters()


In [101]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
        
        attention_mask = torch.tensor([[1 for _ in range(len(inputs[0]))]])
        
        # print(len(inputs[0]))
        # print(len(attention_mask[0]))
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        outs.append(f'{generated_text}, {doc}')
    
    return '\n'.join(outs)

In [86]:
PROMPT = {}
PROMPT['eval_doc_instr'] = '''
You are given an query and a list of document excerpts in the following structure:
Doc1: content
Doc2: content
For each document that contains relevant information to answer the query output: relevant\n 
If a document is not relevant output: irrelevant\n. The output should have the following structure:
Doc1: relevant
Doc2: irrelevent
Do not comment your output, but strictly follow these instructions.
'''

PROMPT['eval_doc_reassur'] = '''
The instructions are clear. I will assess whether the documents contain information that is relevant or irrelevant to answer the question. Please provide me with the query and documents.
'''

PROMPT['eval_doc_ex1'] = '''
Query: Who has the highest goals in world football?
Docs: Doc1: # table : sounds of silence studio album by simon & garfunkel releasedjanuary 17, 1966 ( 1966 - 01 - 17 ) recorded ; april 5 – december 22, 1965 ; (
Doc:  matches considered official internationals by the opposing sides, which would make him the first footballer to score 50 or more international goals, ahead of imre schlosser, and was the fastest to achieve the feat, scoring his 50th goal in his 32nd official international match, with a four - goal haul against hungary on 31 may 1909. [ 17 ] puskas overall scored 84 goals in his international career, [ 11 ] and remained the highest international goalscorer for 24 years following his 84th goal in 1956 against austria, until mokhtar dahari of malaysia broke the record in the merdeka tournament after scoring his 85th goal on 27 october 1980 against kuwait and he went on to score 89 goals for his country in 142 international appearances. [ 18 ] [ 19 ] [ 20 ] in 2004, ali daei of iran broke the record after scoring his 90th goal against lebanon. [ 21 ] [ 22 ] daei also became the first player to score over 100 goals in international football, ending his career with 108 in total. [ 23 ] [ 24 ] his 100th goal came on 17 november 2004, when he scored a four - goal haul against laos in a 2006 fifa world cup qualification match. [ 25 ] however,
'''
PROMPT['eval_doc_answ1'] = '''#relevant'''

PROMPT['eval_doc_ex2'] = '''Query: Who has the highest goals in world football?
Doc:  # table : sounds of silence studio album by simon & garfunkel releasedjanuary 17, 1966 ( 1966 - 01 - 17 ) recorded ; april 5 – december 22, 1965 ; ( 
'''

PROMPT['eval_doc_answ2'] = '''#irrelevant\n'''

PROMPT['eval_doc_ex3'] = '''Query: Who has the highest goals in world football?
Doc:  matches considered official internationals by the opposing sides, which would make him the first footballer to score 50 or more international goals, ahead of imre schlosser, and was the fastest to achieve the feat, scoring his 50th goal in his 32nd official international match, with a four - goal haul against hungary on 31 may 1909. [ 17 ] puskas overall scored 84 goals in his international career, [ 11 ] and remained the highest international goalscorer for 24 years following his 84th goal in 1956 against austria, until mokhtar dahari of malaysia broke the record in the merdeka tournament after scoring his 85th goal on 27 october 1980 against kuwait and he went on to score 89 goals for his country in 142 international appearances. [ 18 ] [ 19 ] [ 20 ] in 2004, ali daei of iran broke the record after scoring his 90th goal against lebanon. [ 21 ] [ 22 ] daei also became the first player to score over 100 goals in international football, ending his career with 108 in total. [ 23 ] [ 24 ] his 100th goal came on 17 november 2004, when he scored a four - goal haul against laos in a 2006 fifa world cup qualification match. [ 25 ] however,
'''

PROMPT['eval_doc_answ3'] = '''#relevant\n''' 

In [87]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    #print(f"Rank {idx} : {doc}")
    docs_full = [f'Doc{i}: {doc}' for i, doc in enumerate(docs)]
    input= f'''
    Query: {query}
    Doc:  {docs_full}
    '''

    messages = [
        {"role":"user", 'content':PROMPT['eval_doc_instr']},
        {"role":"assistant", 'content':PROMPT['eval_doc_reassur']},
        # {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
        # {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
        # {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
        # {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
        {"role":"user", 'content':input}, 
    ]
    #apply tokenizter + generate eval
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = torch.tensor([[1 for _ in range(len(inputs[0]))]])
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [108]:
importlib.reload(prompts)
query = 'Who has the highest goals in world woman football?'
#retriev docs
docs = retrieve_documents(query)
print('\n'.join(docs))

# # # evaluate retrieved docs
# res = evaluate_docs(query, docs)
# print(res)

# # # # table : look for list of fifa women ' s world cup goalscorers on one of wikipedia ' s sister projects : ; wiktionary ( dictionary ) ; wikibooks ( textbooks ) ; wikiquote ( quotations ) ; wikisource ( library ) ; wikiversity ( learning resources ) ; commons ( media ) ; wikivoyage ( travel guide ) ; wikinews ( news source ) ; wikidata ( linked database ) ; wikispecies ( species directory ) ; wikipedia does not have an article with this exact name. please search for list of fifa women ' s world cup goalscorers in wikipedia to check for alternative titles or spellings. ; you need to log in or create an account and be autoconfirmed to create new articles. alternatively, you can use the article wizard to submit a draft for review, or request a new article. ; search for " list of fifa women ' s world cup goalscorers " in existing articles. ; look for pages within wikipedia that link to this title. ; other reasons this message may be displayed : ; if a page was recently created here, i

In [94]:
docs

['ad ) late antiquity ( 284 ad – 500 ad ) * classical greece ( 480 bc – 338 bc ) * macedonian era ( 338 bc – 323 bc ) * hellenistic greece ( 323 bc – 146 bc ) * late roman republic ( 147 bc – 27 bc ) * principate of the roman empire ( 27 bc – 284 ad ) * late antiquity ( 284 ad – 500 ad ) * migration period ( europe, 300 ad – 700 ad ) * middle ages ( europe, 476 – 1453 ) byzantine era ( 330 – 1453 ) early middle ages ( europe, 476 – 1066 ) viking age ( scandinavia, europe, 793 – 1066 ) high middle ages ( europe, 1066 – c. 1300 ) late middle ages ( europe, c. 1300 – 1453 ) the renaissance ( europe, c. 1300 – c. 1601 ) * byzantine era ( 330 – 1453 ) * early middle ages ( europe, 476 – 1066 ) viking age ( scandinavia, europe, 793 – 1066 ) * viking age ( scandinavia, europe, 793 – 1066 ) * high middle ages ( europe, 1066 – c. 1300 ) * late middle ages ( europe, c',
 '– 500 ad ) * migration period ( europe, 300 ad – 700 ad ) * archaic period ( 776 bc – 612 bc ) – the establishment of city - 

In [83]:
rel_docs

'#relevant'

In [68]:
def create_new_query_prompt(query, context, fu_question):
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':input},
        {'role':'assistant', 'content':fu_question}
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt").to(device)
    
    return {'inputs': inputs}   
    

In [41]:
qq_train_df = pd.DataFrame(columns=['question', 'relevant_docs', 'follow_up_questions'])
for idx, entry in enumerate(qq_ds):
    if idx == 1: break
    query = entry['question']
    fu_questions = entry['follow_up_questions']
    
    #retriev docs
    docs = retrieve_documents(query)
    
    # evaluate retrieved docs
    rel_docs = evaluate_docs(query, docs)
    
    qq_train_df.loc[len(qq_train_df)]  = [query, '\n'.join(rel_docs), fu_questions]

# make dataset
train_ds = Dataset.from_pandas(qq_train_df)
    
#Apply the tokenization function to the dataset
train_ds = train_ds.map(
    lambda row: create_new_query_prompt(row['question'], row['relevant_docs'], row['follow_up_questions']), 
    batched=False, 
    remove_columns=train_ds.column_names
)

tensor([132,  53,  12, 179,   0, 115, 182,  36, 178, 264], device='cuda:0')
1214
1214
1239
1239
1240
1240
1222
1222
1041
1041
1214
1214
1232
1232
1215
1215
1236
1236
1216
1216


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

TypeError: Provided `function` which is applied to all elements of table returns a variable of type <class 'torch.Tensor'>. Make sure provided `function` returns a variable of type `dict` (or a pyarrow table) to update the dataset or `None` if you are only interested in side effects.

In [73]:
importlib.reload(prompts)
def make_new_query(query, context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = torch.tensor([[1 for _ in range(len(inputs[0]))]])
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=258)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [103]:

print(make_new_query(query, res))

### To answer the query 'What period came before the period containing the enlightenment?' we need to identify the time period of the enlightenment and then find the previous period.

From the given context information, we can see that the enlightenment is mentioned in the 18th century. Therefore, we need to identify the time period that came before the 18th century.

### Possible follow-up questions to answer the query:

1. What is the time period of the enlightenment?
2. What was the time period before the 18th century?
3. What are the major events and periods that occurred before the enlightenment?
4. How does the enlightenment relate to the previous periods?
5. What are the key characteristics of the period before the enlightenment?

### Possible follow-up questions to narrow down the time period:

1. Was the period before the enlightenment in the classical antiquity (480 BC - 476 AD) era?
2. Was the period before the enlightenment in the medieval period (476 - 1453 AD)?
3. Was the